MST 판단

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import random
import heapq
import math

# 랜덤한 50개 노드 생성 (x, y 좌표 포함)
num_nodes = 5
nodes = {i: (random.uniform(0, 100), random.uniform(0, 100)) for i in range(num_nodes)}

# 그래프 생성
G = nx.Graph()

# 노드 추가 (x, y 좌표를 속성으로 저장)
for node, (x, y) in nodes.items():
    G.add_node(node, pos=(x, y))

# 두 점 사이의 거리 계산 함수
def euclidean_distance(node1, node2):
    x1, y1 = nodes[node1]
    x2, y2 = nodes[node2]
    return math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)

# 가까운 점들과 연결하기 (각 노드당 3~5개의 가까운 노드 연결)
for node in G.nodes():
    distances = [(euclidean_distance(node, other), other) for other in G.nodes() if other != node]
    distances.sort()  # 거리순 정렬
    num_connections = random.randint(3, 5)  # 각 노드당 연결 개수 (3~5개)
    for _, neighbor in distances[:num_connections]:
        weight = euclidean_distance(node, neighbor)
        G.add_edge(node, neighbor, weight=round(weight, 2))  # 가중치는 거리, 소수점 2자리까지 반올림

# 프림 알고리즘을 이용한 최소 신장 트리 (MST) 찾기
def prim_mst(graph):
    start_node = 0  # 시작 노드 선택
    mst = nx.Graph()  # 최소 신장 트리를 저장할 그래프
    visited = set([start_node])  # 방문한 노드
    edges = []  # 가능한 엣지 리스트

    # 시작 노드에서 연결된 엣지를 우선순위 큐에 추가
    for neighbor, edge_data in graph[start_node].items():
        heapq.heappush(edges, (edge_data["weight"], start_node, neighbor))
    while edges and len(visited) < len(graph.nodes):  # 모든 노드가 연결될 때까지
        weight, node1, node2 = heapq.heappop(edges)  # 가중치가 가장 낮은 엣지 선택
        if node2 not in visited:  # 아직 방문하지 않은 노드라면 추가
            visited.add(node2)
            mst.add_edge(node1, node2, weight=weight)
            # 새로 방문한 노드에서 갈 수 있는 간선 추가
            for neighbor, edge_data in graph[node2].items():
                if neighbor not in visited:
                    heapq.heappush(edges, (edge_data["weight"], node2, neighbor))
    return mst

# MST 실행
MST_G = prim_mst(G)

# 그래프 시각화
plt.figure(figsize=(8, 8))
pos = nx.get_node_attributes(G, 'pos')  # 노드 위치 설정

# 기본 그래프
nx.draw(G, pos, with_labels=True, node_color='lightgray', node_size=300, edge_color='gray', alpha=0.5)

# 최적화
nx.draw(MST_G, pos, with_labels=True, node_color='skyblue', node_size=500, edge_color='red', width=2)

# 엣지 가중치 표시
edge_labels = {(u, v): f"{d['weight']:.1f}" for u, v, d in MST_G.edges(data=True)}
nx.draw_networkx_edge_labels(MST_G, pos, edge_labels=edge_labels, font_size=8, font_color="black")

plt.title("Minimum Spanning Tree")
plt.show()

실제 지도 위 지하철 역 찍어보기

In [ ]:
!pip install folium
import folium
import pandas as pd

file_path = "/content/역 경도 위도.xlsx"
df = pd.read_excel(file_path, engine="openpyxl")

print(df.columns.tolist())  # '위도', '경도', '역이름'이 정확한지 확인

# 서울 지도 생성 (서울 시청을 중심으로)
seoul_map = folium.Map(location=[37.5665, 126.9780], zoom_start=11)

# 지하철역 지도 위에 추가 (빨간 원형 마커)
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["위도"], row["경도"]],
        radius=4,
        color="black",
        fill=True,
        fill_color="red",
        fill_opacity=0.8,
        popup=row["역명"]
    ).add_to(seoul_map)

seoul_map


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 데이터 불러오기
file_path = "/content/역 경도 위도.xlsx"
df = pd.read_excel(file_path, engine="openpyxl")  # Excel 파일 읽기

# 흰색 배경 설정
plt.figure(figsize=(10, 10), facecolor="white")

# 점만 표시 (위도, 경도 플롯)
plt.scatter(df["경도"], df["위도"], color="red", s=10)  # 경도(X축), 위도(Y축), 검은색 점

plt.axis("off")
plt.show()

A* 알고리즘 사용해 초기 그래프 생성

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
from scipy.spatial import KDTree
from heapq import heappop, heappush
import matplotlib.pyplot as plt

!pip install openpyxl

df = pd.read_excel("/content/역 경도 위도.xlsx", sheet_name='Sheet1')

coords = list(zip(df['경도'], df['위도']))

tree = KDTree(coords)

G = nx.Graph()
for idx, (lon, lat) in enumerate(coords):
    G.add_node(idx, pos=(lon, lat))

# 인접 노드 연결
k = 5
for idx, (lon, lat) in enumerate(coords):
    _, indices = tree.query((lon, lat), k=k+1)
    for i in range(1, len(indices)):
        neighbor_idx = indices[i]
        distance = np.linalg.norm(np.array(coords[idx]) - np.array(coords[neighbor_idx]))
        G.add_edge(idx, neighbor_idx, weight=distance)

# A* Algorithm
def heuristic(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

def a_star_search(graph, start, goal):
    open_set = []
    heappush(open_set, (0, start))
    came_from = {}
    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start] = 0
    f_score = {node: float('inf') for node in graph.nodes}
    f_score[start] = heuristic(graph.nodes[start]['pos'], graph.nodes[goal]['pos'])

    while open_set:
        _, current = heappop(open_set)
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]

        for neighbor in graph.neighbors(current):
            tentative_g_score = g_score[current] + graph[current][neighbor]['weight']
            if tentative_g_score < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g_score
                f_score[neighbor] = tentative_g_score + heuristic(graph.nodes[neighbor]['pos'], graph.nodes[goal]['pos'])
                heappush(open_set, (f_score[neighbor], neighbor))

    return None

# 모든 쌍의 경로 탐색
paths = []
nodes = list(G.nodes)
for i in range(len(nodes)):
    for j in range(i + 1, len(nodes)):
        path = a_star_search(G, nodes[i], nodes[j])
        if path:
            paths.append(path)

# 시각화
fig, ax = plt.subplots(figsize=(10, 10))
for path in paths:
    path_coords = np.array([G.nodes[node]['pos'] for node in path])
    ax.plot(path_coords[:, 0], path_coords[:, 1], 'b-', alpha=0.5)

for (lon, lat) in coords:
    ax.plot(lon, lat, 'ro', markersize=5)

plt.title("A* Algorithm Subway Network")
plt.axis("off")
plt.show()


MST 이용해 초기 그래프 생성

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import heapq
import random

file_path = "/content/역 경도 위도.xlsx"
df = pd.read_excel(file_path, sheet_name='Sheet1')

coords = list(zip(df['경도'], df['위도']))
num_nodes = len(coords)

G_full = nx.Graph()
for idx, (lon, lat) in enumerate(coords):
    G_full.add_node(idx, pos=(lon, lat))

# 4. 모든 노드 간 거리 계산 및 엣지 추가
for i in range(num_nodes):
    for j in range(i + 1, num_nodes):
        dist = np.linalg.norm(np.array(coords[i]) - np.array(coords[j]))
        G_full.add_edge(i, j, weight=dist)

# 5. Prim 알고리즘
def prim_mst(graph, start_node):
    visited = set()
    mst = nx.Graph()
    pos = nx.get_node_attributes(graph, 'pos')

    for node in graph.nodes:
        mst.add_node(node, pos=pos[node])

    visited.add(start_node)
    edges = []

    for neighbor in graph.neighbors(start_node):
        weight = graph[start_node][neighbor]['weight']
        heapq.heappush(edges, (weight, start_node, neighbor))

    while edges and len(visited) < len(graph.nodes):
        weight, u, v = heapq.heappop(edges)
        if v not in visited:
            visited.add(v)
            mst.add_edge(u, v, weight=weight)
            for neighbor in graph.neighbors(v):
                if neighbor not in visited:
                    new_weight = graph[v][neighbor]['weight']
                    heapq.heappush(edges, (new_weight, v, neighbor))

    return mst

# 6. MST 생성
start_node = random.randint(0, num_nodes - 1)
mst = prim_mst(G_full, start_node)

# 7. 시각화
plt.figure(figsize=(10, 10))
pos = nx.get_node_attributes(G_full, 'pos')

nx.draw_networkx_edges(mst, pos, edge_color='blue', width=1.5, alpha=0.6)
nx.draw_networkx_nodes(G_full, pos, node_color='red', node_size=30)

plt.title("MST Subway Network", fontsize=15)
plt.axis('off')
plt.show()


초기 그래프 기반 부분경로 1개 생성

In [ ]:

import pandas as pd
import networkx as nx
import numpy as np
from scipy.spatial import KDTree
from heapq import heappop, heappush
import matplotlib.pyplot as plt
import random

df = pd.read_excel("/content/역 경도 위도.xlsx", sheet_name='Sheet1')

coords = list(zip(df['경도'], df['위도']))

tree = KDTree(coords)

# 그래프 생성
G = nx.Graph()
for idx, (lon, lat) in enumerate(coords):
    G.add_node(idx, pos=(lon, lat))

# 인접 노드 연결
k = 5
for idx, (lon, lat) in enumerate(coords):
    _, indices = tree.query((lon, lat), k=k+1)
    for i in range(1, len(indices)):
        neighbor_idx = indices[i]
        distance = np.linalg.norm(np.array(coords[idx]) - np.array(coords[neighbor_idx]))
        G.add_edge(idx, neighbor_idx, weight=distance)

# A* 알고리즘
def heuristic(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

def a_star_search(graph, start, goal):
    open_set = []
    heappush(open_set, (0, start))
    came_from = {}
    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start] = 0
    f_score = {node: float('inf') for node in graph.nodes}
    f_score[start] = heuristic(graph.nodes[start]['pos'], graph.nodes[goal]['pos'])

    while open_set:
        _, current = heappop(open_set)
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]

        for neighbor in graph.neighbors(current):
            tentative_g_score = g_score[current] + graph[current][neighbor]['weight']
            if tentative_g_score < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g_score
                f_score[neighbor] = tentative_g_score + heuristic(graph.nodes[neighbor]['pos'], graph.nodes[goal]['pos'])
                heappush(open_set, (f_score[neighbor], neighbor))

    return None

# 모든 쌍의 경로 탐색
paths = []
nodes = list(G.nodes)
for i in range(len(nodes)):
    for j in range(i + 1, len(nodes)):
        path = a_star_search(G, nodes[i], nodes[j])
        if path:
            paths.append(path)

# 랜덤 부분 경로 시각화 (1개)
def visualize_random_paths(graph, all_paths, num_paths=1):
    fig, ax = plt.subplots(figsize=(10, 10))
    random_paths = random.sample(all_paths, num_paths)

    for path in random_paths:
        path_coords = np.array([graph.nodes[node]['pos'] for node in path])
        ax.plot(path_coords[:, 0], path_coords[:, 1], '-', linewidth=2)

        for node in path:
            lon, lat = graph.nodes[node]['pos']
            ax.plot(lon, lat, 'ro', markersize=5)


    plt.axis("off")
    plt.show()

visualize_random_paths(G, paths, num_paths=1)


초기 그래프 기반 부분경로 9개 생성

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
from scipy.spatial import KDTree
from heapq import heappop, heappush
import matplotlib.pyplot as plt
import random


df = pd.read_excel("/content/역 경도 위도.xlsx", sheet_name='Sheet1')

coords = list(zip(df['경도'], df['위도']))

tree = KDTree(coords)

# 그래프 생성
G = nx.Graph()
for idx, (lon, lat) in enumerate(coords):
    G.add_node(idx, pos=(lon, lat))

# 인접 노드 연결
k = 5
for idx, (lon, lat) in enumerate(coords):
    _, indices = tree.query((lon, lat), k=k+1)
    for i in range(1, len(indices)):
        neighbor_idx = indices[i]
        distance = np.linalg.norm(np.array(coords[idx]) - np.array(coords[neighbor_idx]))
        G.add_edge(idx, neighbor_idx, weight=distance)

# A* 알고리즘
def heuristic(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

def a_star_search(graph, start, goal):
    open_set = []
    heappush(open_set, (0, start))
    came_from = {}
    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start] = 0
    f_score = {node: float('inf') for node in graph.nodes}
    f_score[start] = heuristic(graph.nodes[start]['pos'], graph.nodes[goal]['pos'])

    while open_set:
        _, current = heappop(open_set)
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]

        for neighbor in graph.neighbors(current):
            tentative_g_score = g_score[current] + graph[current][neighbor]['weight']
            if tentative_g_score < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g_score
                f_score[neighbor] = tentative_g_score + heuristic(graph.nodes[neighbor]['pos'], graph.nodes[goal]['pos'])
                heappush(open_set, (f_score[neighbor], neighbor))

    return None

# 모든 쌍의 경로 탐색
paths = []
nodes = list(G.nodes)
for i in range(len(nodes)):
    for j in range(i + 1, len(nodes)):
        path = a_star_search(G, nodes[i], nodes[j])
        if path:
            paths.append(path)

# 랜덤 부분 경로 시각화 (9개)
def visualize_random_paths(graph, all_paths, num_paths=1):
    fig, ax = plt.subplots(figsize=(10, 10))
    random_paths = random.sample(all_paths, num_paths)

    for path in random_paths:
        path_coords = np.array([graph.nodes[node]['pos'] for node in path])
        ax.plot(path_coords[:, 0], path_coords[:, 1], '-', linewidth=2)

        for node in path:
            lon, lat = graph.nodes[node]['pos']
            ax.plot(lon, lat, 'ro', markersize=5)


    plt.axis("off")
    plt.show()

visualize_random_paths(G, paths, num_paths=3)

자체 알고리즘 사용해 초기 그래프에 적용

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
from scipy.spatial import KDTree
import random
import matplotlib.pyplot as plt

df = pd.read_excel("/content/역 경도 위도.xlsx", sheet_name='Sheet1')
coords = list(zip(df['경도'], df['위도']))

# KDTree 기반 그래프 G 생성
tree = KDTree(coords)
G = nx.Graph()
for idx, (lon, lat) in enumerate(coords):
    G.add_node(idx, pos=(lon, lat))

k = 5
for idx, (lon, lat) in enumerate(coords):
    _, indices = tree.query((lon, lat), k=k + 1)
    for neighbor in indices[1:]:
        distance = np.linalg.norm(np.array(coords[idx]) - np.array(coords[neighbor]))
        G.add_edge(idx, neighbor, weight=distance)

# 초기 그래프 U 및 관련 집합 정의
U = G.copy()
Edge_U = set(U.edges)
Nod_U = set(U.nodes)

Edge_U_dict = {node: set() for node in U.nodes}
for u, v in Edge_U:
    Edge_U_dict[u].add((u, v))
    Edge_U_dict[v].add((u, v))

# 9개 경로 초기화
num_paths = 9
Edge_list = [set() for _ in range(num_paths)]
Nod_list = [set() for _ in range(num_paths)]
i_current_nod = [None for _ in range(num_paths)]
active_paths = set(range(num_paths))

# 시작 노드 지정
start_nodes = random.sample(list(Nod_U), num_paths)
for i in range(num_paths):
    start = start_nodes[i]
    Nod_list[i].add(start)
    i_current_nod[i] = start
    Nod_U.discard(start)

# 각 Pi 초기 확장
for i in list(active_paths):
    current = i_current_nod[i]
    if Edge_U_dict[current]:
        edge = random.choice(list(Edge_U_dict[current]))
        u, v = edge
        next_node = v if current == u else u
        Edge_list[i].add(edge)
        Edge_U.discard(edge)
        Edge_U_dict[u].discard(edge)
        Edge_U_dict[v].discard(edge)
        Nod_list[i].add(next_node)
        i_current_nod[i] = next_node
        Nod_U.discard(next_node)
    else:
        active_paths.discard(i)

# 경로 확장 반복
while Nod_U and active_paths:
    for i in list(active_paths):
        current = i_current_nod[i]
        if not Edge_U_dict[current]:
            active_paths.discard(i)
            continue
        edge = random.choice(list(Edge_U_dict[current]))
        u, v = edge
        next_node = v if current == u else u
        Edge_list[i].add(edge)
        Edge_U.discard(edge)
        Edge_U_dict[u].discard(edge)
        Edge_U_dict[v].discard(edge)
        Nod_list[i].add(next_node)
        i_current_nod[i] = next_node
        Nod_U.discard(next_node)

# 미포함 노드를 기존 G의 간선으로 연결
while Nod_U:
    remaining = list(Nod_U)
    for node in remaining:
        connected_edges = list(G.edges(node))
        random.shuffle(connected_edges)
        found = False
        for u, v in connected_edges:
            for i in range(num_paths):
                if u in Nod_list[i] or v in Nod_list[i]:
                    Edge_list[i].add((u, v))
                    Nod_list[i].add(node)
                    Nod_U.discard(node)
                    found = True
                    break
            if found:
                break
        if not found:
            Nod_U.discard(node)

# 시각화
fig, ax = plt.subplots(figsize=(10, 10))
colors = plt.cm.get_cmap("tab10", num_paths)

for i in range(num_paths):
    for edge in Edge_list[i]:
        x = [G.nodes[edge[0]]['pos'][0], G.nodes[edge[1]]['pos'][0]]
        y = [G.nodes[edge[0]]['pos'][1], G.nodes[edge[1]]['pos'][1]]
        ax.plot(x, y, color=colors(i), linewidth=2)

    for node in Nod_list[i]:
        lon, lat = G.nodes[node]['pos']
        ax.plot(lon, lat, 'o', color=colors(i), markersize=4)

plt.axis("off")
plt.show()


엣지 데이터 생성 및 예시 적용 코드

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations
from collections import deque

!pip install openpyxl

file_path = "/content/역 경도 위도.xlsx"
edge_path = "/content/엣지 데이터.csv"

df = pd.read_excel(file_path)
coords = list(zip(df['경도'], df['위도']))
edges_df = pd.read_csv(edge_path)


G = nx.Graph()
for _, row in edges_df.iterrows():
    u, v = int(row['from']), int(row['to'])
    G.add_node(u, pos=(row['from_lon'], row['from_lat']))
    G.add_node(v, pos=(row['to_lon'], row['to_lat']))
    G.add_edge(u, v, weight=row['weight'])

name_to_index = {name: idx for idx, name in enumerate(df['역명'])}

start = name_to_index['남태령']
transfer = name_to_index['서울']
end = name_to_index['장암']

# 경로 탐색 함수 설정
def bfs_path_no_angle(graph, start, end, max_len=30):
    queue = deque()
    queue.append((start, [start]))
    while queue:
        current, path = queue.popleft()
        if len(path) > max_len:
            continue
        if current == end:
            return path
        for neighbor in graph.neighbors(current):
            if neighbor not in path:
                queue.append((neighbor, path + [neighbor]))
    return None

# 부분 경로 탐색
path1 = bfs_path_no_angle(G, start, transfer, max_len=15)
path2 = bfs_path_no_angle(G, transfer, end, max_len=15)

# 경로 병합
full_path = None
if path1 and path2:
    full_path = path1[:-1] + path2

fig, ax = plt.subplots(figsize=(12, 12))
if full_path:
    path_coords = np.array([G.nodes[node]['pos'] for node in full_path])
    ax.plot(path_coords[:, 0], path_coords[:, 1], '-', color='green', linewidth=3)

for (lon, lat) in coords:
    ax.plot(lon, lat, 'k.', markersize=3)

plt.axis("off")
plt.legend()
plt.show()


필요해보이는 노드 그룹화 , 2호선 순환노선 생성 후 나머지 미포함 노드 연결

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from scipy.spatial import KDTree

file_path = "/content/역 경도 위도.xlsx"
df = pd.read_excel(file_path, sheet_name='Sheet1')
df = df[df['역명'] != '독바위']

station_to_coord = dict(zip(df['역명'], zip(df['경도'], df['위도'])))
all_station_names = set(df['역명'])

# 그룹화 노선 정의
station_groups = [
    ['지축', '구파발', '연신내', '구산', '응암', '새절', '증산', '디지털미디어시티', '월드컵경기장', '마포구청', '망원', '합정'],
    ['역촌','불광', '녹번', '홍제', '무악재', '독립문'],
    ['방화','개화산','김포공항','송정','발산', '우장산', '화곡', '까치산', '신정'],
    ['개화','김포공항','공항시장','신방화','마곡나루', '양천향교', '가양', '증미', '등촌', '염창', '신목동', '선유도', '당산'],
    ['온수', '천왕', '광명사거리', '철산', '가산디지털단지', '남구로', '대림'],
    ['남태령', '사당'],
    ['모란', '수진', '신흥', '단대오거리', '남한산성입구', '산성', '남위례', '복정', '장지', '문정', '가락시장'],
    ['마천', '거여', '개롱'],
    ['하남검단산', '하남시청', '하남풍산', '미사', '강일', '상일동', '고덕', '명일'],
    ['장암', '도봉산', '수락산', '마들', '노원', '중계', '하계', '공릉', '태릉입구', '먹골', '중화', '상봉', '면목', '사가정', '용마산', '중곡', '군자'],
    ['당고개', '상계', '노원', '창동', '쌍문', '수유', '미아', '미아사거리', '길음'],
    ['신내', '봉화산', '화랑대', '태릉입구', '석계', '돌곶이', '상월곡']
]
grouped_stations = set(station for group in station_groups for station in group)

# KDTree에 사용할 역들 추가 목록
include_stations = ['길음', '상월곡', '군자', '명일', '개롱', '가락시장',
                    '사당', '대림', '까치산', '당산', '합정', '독립문','신정']

# 제외할 노선
excluded_group = ['방화','개화산','김포공항','송정','발산', '우장산', '화곡', '까치산']
excluded_group_set = set(excluded_group)

# KDTree 연결 대상 역
final_target_stations = [
    s for s in (all_station_names - grouped_stations).union(include_stations)
    if s in station_to_coord and s not in excluded_group_set
]
final_coords = [station_to_coord[s] for s in final_target_stations]

# KDTree 생성 및 간선 연결
tree_final_clean = KDTree(final_coords)
G_final_clean = nx.Graph()
for s in final_target_stations:
    G_final_clean.add_node(s, pos=station_to_coord[s])

k = 5
threshold = 0.04
for i, station in enumerate(final_target_stations):
    _, indices = tree_final_clean.query(station_to_coord[station], k=k+1)
    for j in indices[1:]:
        neighbor = final_target_stations[j]
        dist = np.linalg.norm(np.array(station_to_coord[station]) - np.array(station_to_coord[neighbor]))
        if dist <= threshold and not G_final_clean.has_edge(station, neighbor):
            G_final_clean.add_edge(station, neighbor, weight=dist)

# 2호선 역 그룹화
line_2 = [
    '시청', '을지로입구', '을지로3가', '을지로4가', '동대문역사문화공원', '신당', '상왕십리', '왕십리',
    '한양대', '뚝섬', '성수', '건대입구', '구의', '강변', '잠실나루', '잠실', '잠실새내', '종합운동장',
    '삼성', '선릉', '역삼', '강남', '교대', '서초', '방배', '사당', '낙성대', '서울대입구', '봉천',
    '신림', '신대방', '구로디지털단지', '대림', '신도림', '문래', '영등포구청', '당산',
    '합정', '홍대입구', '신촌', '이대', '아현', '충정로', '시청'  # 순환
]
line_2_filtered = [s for s in line_2 if s in station_to_coord]
for i in range(len(line_2_filtered) - 1):
    u, v = line_2_filtered[i], line_2_filtered[i + 1]
    dist = np.linalg.norm(np.array(station_to_coord[u]) - np.array(station_to_coord[v]))
    G_final_clean.add_edge(u, v, weight=dist)

fig, ax = plt.subplots(figsize=(12, 12))

# 그룹화 노선
colors = plt.cm.get_cmap('tab10', len(station_groups))
for idx, group in enumerate(station_groups):
    group = [s for s in group if s in station_to_coord]
    for i in range(len(group)-1):
        u, v = group[i], group[i+1]
        x = [station_to_coord[u][0], station_to_coord[v][0]]
        y = [station_to_coord[u][1], station_to_coord[v][1]]
        ax.plot(x, y, '-', color=colors(idx), linewidth=2)

# KDTree 연결(그룹화 안된 나머지 노드 및 추가 목록 역들)
for u, v in G_final_clean.edges:
    if {u, v}.issubset(set(line_2_filtered)):
        continue
    x = [station_to_coord[u][0], station_to_coord[v][0]]
    y = [station_to_coord[u][1], station_to_coord[v][1]]
    ax.plot(x, y, '-', color='gray', linewidth=1.5)

# 2호선 시각화
for i in range(len(line_2_filtered)-1):
    u, v = line_2_filtered[i], line_2_filtered[i+1]
    x = [station_to_coord[u][0], station_to_coord[v][0]]
    y = [station_to_coord[u][1], station_to_coord[v][1]]
    ax.plot(x, y, '-', color='green', linewidth=3)


for name, (lon, lat) in station_to_coord.items():
    ax.plot(lon, lat, 'ko', markersize=3)


plt.axis("off")
plt.show()


경로 생성 함수 정의, 6개 노선 추가 생성

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from scipy.spatial import KDTree

file_path = "/content/역 경도 위도.xlsx"
df = pd.read_excel(file_path, sheet_name='Sheet1')
df = df[df['역명'] != '독바위']
station_to_coord = dict(zip(df['역명'], zip(df['경도'], df['위도'])))

edge_file_path = "/content/엣지 데이터 그룹화.xlsx"
edges_df = pd.read_excel(edge_file_path)

G_grouped = nx.Graph()
for _, row in edges_df.iterrows():
    G_grouped.add_node(row['from'], pos=(row['from_lon'], row['from_lat']))
    G_grouped.add_node(row['to'], pos=(row['to_lon'], row['to_lat']))
    G_grouped.add_edge(row['from'], row['to'], weight=row['weight'])

pos = nx.get_node_attributes(G_grouped, 'pos')

# 6쌍 추출(먼 거리 우선)
custom_terminals = ['합정', '독립문', '신정', '당산', '대림', '사당',
                    '가락시장', '개롱', '명일', '군자', '길음', '상월곡']

all_distances = []
for i in range(len(custom_terminals)):
    for j in range(i+1, len(custom_terminals)):
        s1, s2 = custom_terminals[i], custom_terminals[j]
        dist = np.linalg.norm(np.array(pos[s1]) - np.array(pos[s2]))
        all_distances.append(((s1, s2), dist))

sorted_pairs = sorted(all_distances, key=lambda x: -x[1])
selected_pairs = []
used_stations = set()
for (u, v), _ in sorted_pairs:
    if u not in used_stations and v not in used_stations:
        selected_pairs.append((u, v))
        used_stations.update([u, v])
    if len(selected_pairs) == 6:
        break

station_groups = [
    ['지축', '구파발', '연신내', '구산', '응암', '새절', '증산', '디지털미디어시티', '월드컵경기장', '마포구청', '망원', '합정'],
    ['역촌','불광', '녹번', '홍제', '무악재', '독립문'],
    ['방화','개화산','김포공항','송정','발산', '우장산', '화곡', '까치산', '신정'],
    ['개화','김포공항','공항시장','신방화','마곡나루', '양천향교', '가양', '증미', '등촌', '염창', '신목동', '선유도', '당산'],
    ['온수', '천왕', '광명사거리', '철산', '가산디지털단지', '남구로', '대림'],
    ['남태령', '사당'],
    ['모란', '수진', '신흥', '단대오거리', '남한산성입구', '산성', '남위례', '복정', '장지', '문정', '가락시장'],
    ['마천', '거여', '개롱'],
    ['하남검단산', '하남시청', '하남풍산', '미사', '강일', '상일동', '고덕', '명일'],
    ['장암', '도봉산', '수락산', '마들', '노원', '중계', '하계', '공릉', '태릉입구', '먹골', '중화', '상봉', '면목', '사가정', '용마산', '중곡', '군자'],
    ['당고개', '상계', '노원', '창동', '쌍문', '수유', '미아', '미아사거리', '길음'],
    ['신내', '봉화산', '화랑대', '태릉입구', '석계', '돌곶이', '상월곡']
]
grouped_edges = set()
for group in station_groups:
    for i in range(len(group) - 1):
        u, v = group[i], group[i + 1]
        if u in G_grouped and v in G_grouped:
            grouped_edges.add(frozenset((u, v)))

# 경로 생성 함수
def generate_path_excluding_group(G, start, end, excluded_edges, max_retry=100):
    for _ in range(max_retry):
        path = [start]
        visited = set([start])
        current = start

        while current != end:
            neighbors = list(G.neighbors(current))
            np.random.shuffle(neighbors)
            for neighbor in neighbors:
                edge = frozenset((current, neighbor))
                if neighbor not in visited and edge not in excluded_edges:
                    path.append(neighbor)
                    visited.add(neighbor)
                    current = neighbor
                    break
            else:
                break
        if path[-1] == end:
            return path
    return None

# 6개 노선 생성
lines_no_group = []
for start, end in selected_pairs:
    path = generate_path_excluding_group(G_grouped, start, end, grouped_edges)
    if path:
        lines_no_group.append(path)

# 시각화 함수 정의
def draw_individual_line(line_path, color, title):
    fig, ax = plt.subplots(figsize=(10, 10))
    nx.draw_networkx_edges(G_grouped, pos, ax=ax, edge_color='lightgray', alpha=0.3, width=1)
    nx.draw_networkx_nodes(G_grouped, pos, ax=ax, node_size=8, node_color='gray', alpha=0.4)
    edge_list = [(line_path[i], line_path[i+1]) for i in range(len(line_path)-1)]
    nx.draw_networkx_edges(G_grouped, pos, edgelist=edge_list, edge_color=color, width=3, ax=ax)
    nx.draw_networkx_nodes(G_grouped, pos, nodelist=line_path, node_size=40, node_color=color, ax=ax)
    ax.text(*pos[line_path[0]], line_path[0], fontsize=10, color='black', ha='right', weight='bold')
    ax.text(*pos[line_path[-1]], line_path[-1], fontsize=10, color='black', ha='left', weight='bold')
    plt.title(title, fontsize=14)
    plt.axis("off")
    plt.show()

colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown']
for idx, line_path in enumerate(lines_no_group):
    draw_individual_line(line_path, colors[idx], f"Line {idx+1}")

# 6개의 전체 노선 시각화
fig, ax = plt.subplots(figsize=(10, 10))
nx.draw_networkx_edges(G_grouped, pos, ax=ax, edge_color='lightgray', alpha=0.3, width=1)
nx.draw_networkx_nodes(G_grouped, pos, ax=ax, node_size=8, node_color='gray', alpha=0.4)

for idx, path in enumerate(lines_no_group):
    edge_list = [(path[i], path[i+1]) for i in range(len(path)-1)]
    nx.draw_networkx_edges(G_grouped, pos, edgelist=edge_list, edge_color=colors[idx], width=2.5, ax=ax, label=f"Line {idx+1}")
    nx.draw_networkx_nodes(G_grouped, pos, nodelist=path, node_size=30, node_color=colors[idx], ax=ax)
    ax.text(*pos[path[0]], path[0], fontsize=9, color='black', ha='right', weight='bold')
    ax.text(*pos[path[-1]], path[-1], fontsize=9, color='black', ha='left', weight='bold')

plt.title("Line 1~6", fontsize=10)
plt.axis("off")
plt.legend()
plt.show()


둔각 삼각형 실험

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
from scipy.spatial import KDTree
from heapq import heappop, heappush
import matplotlib.pyplot as plt
from itertools import combinations

!pip install openpyxl


df = pd.read_excel("/content/둔각실험.xlsx", sheet_name='Sheet1')
coords = list(zip(df['경도'], df['위도']))
tree = KDTree(coords)
G = nx.Graph()
for idx, (lon, lat) in enumerate(coords):
    G.add_node(idx, pos=(lon, lat))

# 기존 하던대로 A* 그래프 생성
k = 2
for idx, (lon, lat) in enumerate(coords):
    _, indices = tree.query((lon, lat), k=k+1)
    for i in range(1, len(indices)):
        neighbor_idx = indices[i]
        distance = np.linalg.norm(np.array(coords[idx]) - np.array(coords[neighbor_idx]))
        G.add_edge(idx, neighbor_idx, weight=distance)


def heuristic(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

def a_star_search(graph, start, goal):
    open_set = []
    heappush(open_set, (0, start))
    came_from = {}
    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start] = 0
    f_score = {node: float('inf') for node in graph.nodes}
    f_score[start] = heuristic(graph.nodes[start]['pos'], graph.nodes[goal]['pos'])

    while open_set:
        _, current = heappop(open_set)
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]

        for neighbor in graph.neighbors(current):
            tentative_g_score = g_score[current] + graph[current][neighbor]['weight']
            if tentative_g_score < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g_score
                f_score[neighbor] = tentative_g_score + heuristic(graph.nodes[neighbor]['pos'], graph.nodes[goal]['pos'])
                heappush(open_set, (f_score[neighbor], neighbor))

    return None


paths = []
nodes = list(G.nodes)
for i in range(len(nodes)):
    for j in range(i + 1, len(nodes)):
        path = a_star_search(G, nodes[i], nodes[j])
        if path:
            paths.append(path)

# 둔각 삼각형 판별 후 빗변 제거
def is_obtuse(p1, p2, p3):
    a = np.linalg.norm(np.array(p1) - np.array(p2))
    b = np.linalg.norm(np.array(p2) - np.array(p3))
    c = np.linalg.norm(np.array(p3) - np.array(p1))
    sides = sorted([a, b, c])
    return sides[0]**2 + sides[1]**2 < sides[2]**2

obtuse_edges_to_remove = set()
for triangle in combinations(nodes, 3):
    p1 = G.nodes[triangle[0]]['pos']
    p2 = G.nodes[triangle[1]]['pos']
    p3 = G.nodes[triangle[2]]['pos']
    if is_obtuse(p1, p2, p3):
        distances = {
            (triangle[0], triangle[1]): np.linalg.norm(np.array(p1) - np.array(p2)),
            (triangle[1], triangle[2]): np.linalg.norm(np.array(p2) - np.array(p3)),
            (triangle[2], triangle[0]): np.linalg.norm(np.array(p3) - np.array(p1))
        }
        max_edge = max(distances, key=distances.get)
        obtuse_edges_to_remove.add(tuple(sorted(max_edge)))

for u, v in obtuse_edges_to_remove:
    if G.has_edge(u, v):
        G.remove_edge(u, v)

fig, ax = plt.subplots(figsize=(10, 10))

for u, v in G.edges():
    x1, y1 = G.nodes[u]['pos']
    x2, y2 = G.nodes[v]['pos']
    ax.plot([x1, x2], [y1, y2], 'g-', alpha=0.5)

for (lon, lat) in coords:
    ax.plot(lon, lat, 'ro', markersize=5)

plt.axis("off")
plt.show()

둔각 적용 코드

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from heapq import heappush, heappop
import random

!pip install openpyxl

df_edges = pd.read_excel("/content/엣지데이터 신규.xlsx")
df_nodes = pd.read_excel("/content/역 경도 위도.xlsx", sheet_name="Sheet1")

G = nx.Graph()
for _, row in df_edges.iterrows():
    start = row['from']
    end = row['to']
    start_pos = (row['from_lon'], row['from_lat'])
    end_pos = (row['to_lon'], row['to_lat'])
    dist = row['distance']
    G.add_node(start, pos=start_pos)
    G.add_node(end, pos=end_pos)
    G.add_edge(start, end, weight=dist)

def heuristic(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

# 엣지 재사용 하지 않도록 설정
def random_a_star(graph, start, goal, used_edges):
    open_set = []
    heappush(open_set, (0, start))
    came_from = {}
    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start] = 0
    f_score = {node: float('inf') for node in graph.nodes}
    f_score[start] = heuristic(graph.nodes[start]['pos'], graph.nodes[goal]['pos'])

    while open_set:
        _, current = heappop(open_set)
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]

        neighbors = list(graph.neighbors(current))
        random.shuffle(neighbors)

        for neighbor in neighbors:
            edge = tuple(sorted((current, neighbor)))
            if edge in used_edges:
                continue
            tentative_g = g_score[current] + graph[current][neighbor]['weight']
            if tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f_score[neighbor] = tentative_g + heuristic(graph.nodes[neighbor]['pos'], graph.nodes[goal]['pos'])
                heappush(open_set, (f_score[neighbor], neighbor))
    return None

# 양 끝점 후보 16역 선정
terminal_station_names = ['지축', '독바위', '역곡', '방화', '천왕', '남태령','회룡'
                          ,'중앙보훈병원','모란', '마천', '하남검단산', '수락산', '불암산', '신내','암사','개화']
valid_terminal_indices = []
for name in terminal_station_names:
    matched = df_nodes[df_nodes['역명'] == name]
    if matched.empty:
        continue
    coord = (matched.iloc[0]['경도'], matched.iloc[0]['위도'])
    for node in G.nodes:
        if np.allclose(G.nodes[node]['pos'], coord, atol=1e-5):
            valid_terminal_indices.append(node)
            break

# 16개의 역들 중 랜덤으로 2개씩 총 8쌍 생성
random.shuffle(valid_terminal_indices)
terminal_pairs = [(valid_terminal_indices[i], valid_terminal_indices[i + 1]) for i in range(0, 16, 2)]


# 경로 찾기
used_edges = set()
line_paths = []
for i, (start, goal) in enumerate(terminal_pairs):
    path = random_a_star(G, start, goal, used_edges)
    if path:
        for u, v in zip(path[:-1], path[1:]):
            used_edges.add(tuple(sorted((u, v))))
        line_paths.append((i, path))


# 8개의 호선 시각화
fig, ax = plt.subplots(figsize=(12, 12))
colors = ['red', 'blue', 'yellow', 'orange', 'purple', 'brown','black','pink']
for i, path in line_paths:
    for u, v in zip(path[:-1], path[1:]):
        x1, y1 = G.nodes[u]['pos']
        x2, y2 = G.nodes[v]['pos']
        ax.plot([x1, x2], [y1, y2], color=colors[i], linewidth=2.5)

for _, data in G.nodes(data=True):
    ax.plot(data['pos'][0], data['pos'][1], 'k.', markersize=3)

for idx in valid_terminal_indices:
    x, y = G.nodes[idx]['pos']
    ax.plot(x, y, 'cyan', marker='*', markersize=12)

plt.title("8 random", fontsize=14)
plt.axis('off')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from heapq import heappush, heappop
import random

!pip install openpyxl

df_edges = pd.read_excel("/content/엣지데이터 신규.xlsx")
df_nodes = pd.read_excel("/content/역 경도 위도.xlsx", sheet_name="Sheet1")


G = nx.Graph()
for _, row in df_edges.iterrows():
    start = row['from']
    end = row['to']
    start_pos = (row['from_lon'], row['from_lat'])
    end_pos = (row['to_lon'], row['to_lat'])

    dist = row['distance']

    # distance에 0.1 ~ 1.9 랜덤값 곱해서 다양성 증가
    random_dist = dist * random.uniform(0.1, 1.9)
    G.add_node(start, pos=start_pos)
    G.add_node(end, pos=end_pos)
    G.add_edge(start, end, weight=random_dist)



def heuristic(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

# 엣지 재사용 하지 않도록 설정
def random_a_star(graph, start, goal, used_edges):
    open_set = []
    heappush(open_set, (0, start))
    came_from = {}
    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start] = 0
    f_score = {node: float('inf') for node in graph.nodes}
    f_score[start] = heuristic(graph.nodes[start]['pos'], graph.nodes[goal]['pos'])

    while open_set:
        _, current = heappop(open_set)
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]

        neighbors = list(graph.neighbors(current))
        random.shuffle(neighbors)

        for neighbor in neighbors:
            edge = tuple(sorted((current, neighbor)))
            if edge in used_edges:
                continue
            tentative_g = g_score[current] + graph[current][neighbor]['weight']
            if tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f_score[neighbor] = tentative_g + heuristic(graph.nodes[neighbor]['pos'], graph.nodes[goal]['pos'])
                heappush(open_set, (f_score[neighbor], neighbor))
    return None

# 기존의 2호선 역 및 노선 추가
line2_station_names = ['시청', '을지로입구', '을지로3가', '을지로4가', '동대문역사문화공원',
                       '신당', '상왕십리', '왕십리', '한양대', '뚝섬', '성수', '건대입구',
                       '구의', '강변', '잠실나루', '잠실', '잠실새내', '종합운동장', '삼성',
                       '선릉', '역삼', '강남', '교대', '서초', '방배', '사당', '낙성대',
                       '서울대입구', '봉천', '신림', '신대방', '구로디지털단지', '대림',
                       '신도림', '문래', '영등포구청', '당산', '합정', '홍대입구', '신촌',
                       '이대', '아현', '충정로', '시청']


line2_nodes = [name for name in line2_station_names if name in G.nodes]
line2_full_path = []
line2_used_edges = set()

for u, v in zip(line2_nodes[:-1], line2_nodes[1:]):
    if G.has_edge(u, v):
        line2_used_edges.add(tuple(sorted((u, v))))
    else:
        pos_u = G.nodes[u]['pos']
        pos_v = G.nodes[v]['pos']
        dist = np.linalg.norm(np.array(pos_u) - np.array(pos_v))
        G.add_edge(u, v, weight=dist)
        line2_used_edges.add(tuple(sorted((u, v))))
    if not line2_full_path or line2_full_path[-1] != u:
        line2_full_path.extend([u, v])
    else:
        line2_full_path.append(v)

# 사용된 엣지를 저장
used_edges = set(line2_used_edges)

# 양 끝점 후보 16역 선정
terminal_station_names = ['지축', '독바위', '역곡', '방화', '천왕', '남태령',
                         '회룡', '중앙보훈병원', '모란', '마천', '하남검단산',
                         '수락산', '불암산', '신내', '암사', '개화']
valid_terminal_indices = []
for name in terminal_station_names:
    matched = df_nodes[df_nodes['역명'] == name]
    if matched.empty:
        continue
    coord = (matched.iloc[0]['경도'], matched.iloc[0]['위도'])
    for node in G.nodes:
        if np.allclose(G.nodes[node]['pos'], coord, atol=1e-5):
            valid_terminal_indices.append(node)
            break

# 직접 양 끝점 선별 후 쌍 생성
custom_terminal_pairs = [
    ('천왕', '지축'),
    ('역곡', '회룡'),
    ('하남검단산', '방화'),
    ('독바위', '신내'),
    ('불암산', '남태령'),
    ('개화', '중앙보훈병원'),
    ('수락산', '마천'),
    ('암사', '모란'),
]
# 노선도가 생성될 때 항상 위 순서대로 진행됨을 방지하기 위해 순서 랜덤으로 설정 (이로 인해 다양성 증가)
random.shuffle(custom_terminal_pairs)

# 경로 찾기
line_paths = []
for i, (start, goal) in enumerate(custom_terminal_pairs):
    path = random_a_star(G, start, goal, used_edges)
    if path:
        for u, v in zip(path[:-1], path[1:]):
            used_edges.add(tuple(sorted((u, v))))
        line_paths.append((i, path))

fig, ax = plt.subplots(figsize=(12, 12))
colors = ['red', 'blue', 'yellow', 'orange', 'purple', 'brown','black','pink']

# 8개 노선 시각화
for i, path in line_paths:
    for u, v in zip(path[:-1], path[1:]):
        x1, y1 = G.nodes[u]['pos']
        x2, y2 = G.nodes[v]['pos']
        ax.plot([x1, x2], [y1, y2], color=colors[i], linewidth=2.5, label=f'Line {i+1}' if u == path[0] else "")


for u, v in zip(line2_full_path[:-1], line2_full_path[1:]):
    x1, y1 = G.nodes[u]['pos']
    x2, y2 = G.nodes[v]['pos']
    ax.plot([x1, x2], [y1, y2], color='green', linewidth=2.5, label='Line 2' if u == line2_full_path[0] else "")


for _, data in G.nodes(data=True):
    ax.plot(data['pos'][0], data['pos'][1], 'k.', markersize=3)


for idx in valid_terminal_indices:
    x, y = G.nodes[idx]['pos']
    ax.plot(x, y, 'cyan', marker='*', markersize=12)

plt.title("9 Line", fontsize=10)
plt.axis('off')
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import random
from heapq import heappush, heappop


!pip install openpyxl

file_path_edge = "/content/엣지데이터 신규.xlsx"
file_path_node = "/content/역 경도 위도.xlsx"

df_edges = pd.read_excel(file_path_edge)
df_nodes = pd.read_excel(file_path_node, sheet_name="Sheet1")

G_base = nx.Graph()
for _, row in df_edges.iterrows():
    start, end = row['from'], row['to']
    start_pos, end_pos = (row['from_lon'], row['from_lat']), (row['to_lon'], row['to_lat'])
    dist = row['distance']
    G_base.add_node(start, pos=start_pos)
    G_base.add_node(end, pos=end_pos)
    G_base.add_edge(start, end, weight=dist)

def heuristic(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

def random_a_star(graph, start, goal, used_edges):
    open_set = []
    heappush(open_set, (0, start))
    came_from = {}
    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start] = 0
    f_score = {node: float('inf') for node in graph.nodes}
    f_score[start] = heuristic(graph.nodes[start]['pos'], graph.nodes[goal]['pos'])
    while open_set:
        _, current = heappop(open_set)
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]
        neighbors = list(graph.neighbors(current))
        random.shuffle(neighbors)
        for neighbor in neighbors:
            edge = tuple(sorted((current, neighbor)))
            if edge in used_edges:
                continue
            tentative_g = g_score[current] + graph[current][neighbor]['weight']
            if tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f_score[neighbor] = tentative_g + heuristic(graph.nodes[neighbor]['pos'], graph.nodes[goal]['pos'])
                heappush(open_set, (f_score[neighbor], neighbor))
    return None

line2_station_names = ['시청', '을지로입구', '을지로3가', '을지로4가', '동대문역사문화공원',
                       '신당', '상왕십리', '왕십리', '한양대', '뚝섬', '성수', '건대입구',
                       '구의', '강변', '잠실나루', '잠실', '잠실새내', '종합운동장', '삼성',
                       '선릉', '역삼', '강남', '교대', '서초', '방배', '사당', '낙성대',
                       '서울대입구', '봉천', '신림', '신대방', '구로디지털단지', '대림',
                       '신도림', '문래', '영등포구청', '당산', '합정', '홍대입구', '신촌',
                       '이대', '아현', '충정로', '시청']

line2_nodes = [name for name in line2_station_names if name in G_base.nodes]
line2_full_path, line2_used_edges = [], set()

for u, v in zip(line2_nodes[:-1], line2_nodes[1:]):
    if not G_base.has_edge(u, v):
        dist = np.linalg.norm(np.array(G_base.nodes[u]['pos']) - np.array(G_base.nodes[v]['pos']))
        G_base.add_edge(u, v, weight=dist)
    line2_used_edges.add(tuple(sorted((u, v))))
    if not line2_full_path or line2_full_path[-1] != u:
        line2_full_path.extend([u, v])
    else:
        line2_full_path.append(v)

custom_terminal_pairs = [
    ('천왕', '지축'),
    ('역곡', '회룡'),
    ('하남검단산', '방화'),
    ('독바위', '신내'),
    ('불암산', '남태령'),
    ('개화', '중앙보훈병원'),
    ('수락산', '마천'),
    ('암사', '모란'),
]

# 적합도 함수 설정
def calculate_fitness(line_paths, G, total_nodes):
    visited_nodes = set()
    for _, path in line_paths:
        visited_nodes.update(path)
    node_score = len(visited_nodes) / total_nodes * 100
    return node_score, total_nodes - len(visited_nodes)

# 개체군 생성
population = []
for _ in range(1000):
    G = G_base.copy()
    for u, v, data in G.edges(data=True):
        original_dist = np.linalg.norm(np.array(G.nodes[u]['pos']) - np.array(G.nodes[v]['pos']))
        G[u][v]['weight'] = original_dist * random.uniform(0.1, 1.9) # 개체군을 생성 할때마다 새로운 랜덤값 설정으로 다양성 증가

    random.shuffle(custom_terminal_pairs)
    line_paths = [(100, line2_full_path)]
    used_edges_copy = set(line2_used_edges)

    for i, (start, goal) in enumerate(custom_terminal_pairs):
        path = random_a_star(G, start, goal, used_edges_copy)
        if path:
            for u, v in zip(path[:-1], path[1:]):
                used_edges_copy.add(tuple(sorted((u, v))))
            line_paths.append((i, path))

    population.append(line_paths)

# 적합도 평가
fitness_results = [calculate_fitness(ind, G, len(G_base.nodes)) for ind in population]
fitness_scores = [score for score, _ in fitness_results]
top_10_indices = np.argsort(fitness_scores)[-10:][::-1]

print("Top 10 Node Coverage Results")
for rank, idx in enumerate(top_10_indices, 1):
    score, unvisited = fitness_results[idx]
    print(f"{rank}등 포함률: {score:.2f}% | 미포함 노드 수: {unvisited}")

colors = ['green', 'blue', 'yellow', 'orange', 'purple', 'brown', 'black', 'pink', 'gray', 'cyan']

for rank, idx in enumerate(top_10_indices, 1):
    solution = population[idx]
    fig, ax = plt.subplots(figsize=(5,5))
    for i, path in solution:
        for u, v in zip(path[:-1], path[1:]):
            x1, y1 = G_base.nodes[u]['pos']
            x2, y2 = G_base.nodes[v]['pos']
            ax.plot([x1, x2], [y1, y2], color=colors[i % len(colors)], linewidth=2.5, label=f'Line {i+1}' if u == path[0] else "")
    for _, data in G_base.nodes(data=True):
        ax.plot(data['pos'][0], data['pos'][1], 'k.', markersize=3)
    plt.title(f"Rank {rank} Node Coverage", fontsize=5)
    plt.axis('off')
    plt.legend()
    plt.show()


강남역 예시

In [ ]:
!pip install osmnx geopandas --quiet

import osmnx as ox
import pandas as pd

# 강남역 기준
station_name = "Gangnam Station, Seoul, South Korea"
lat, lon = 37.497175, 127.027926

# 주요 시설 태그 정리
tags = {
    'amenity': ['school', 'hospital', 'university', 'library'],
    'shop': True,
    'leisure': ['park']
}

# OSM에서 정보 수집 (반경 500m)
pois = ox.features_from_point((lat, lon), tags=tags, dist=500)

# 시설 종류별 개수 집계
facility_count = {
    '역명': '강남',
    '병원수': pois[pois['amenity'] == 'hospital'].shape[0],
    '학교수': pois[pois['amenity'] == 'school'].shape[0],
    '대학교수': pois[pois['amenity'] == 'university'].shape[0],
    '도서관수': pois[pois['amenity'] == 'library'].shape[0],
    '상점수': pois[pois['shop'].notnull()].shape[0],
    '공원수': pois[pois['leisure'] == 'park'].shape[0],
}

# 결과 출력
facility_df = pd.DataFrame([facility_count])
facility_df

Simulated Annealing 알고리즘 적용

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import random
import copy
from heapq import heappush, heappop

file_path_edge = "/content/엣지데이터 신규.xlsx"
file_path_node = "/content/역 경도 위도.xlsx"
file_path_od_weight = "/content/쌍가중치_정규화_OD.xlsx"
file_path_node_score = "/content/총점수_정규화_역주변시설.xlsx"

df_edges = pd.read_excel(file_path_edge)
df_nodes = pd.read_excel(file_path_node, sheet_name="Sheet1")
df_od = pd.read_excel(file_path_od_weight).dropna(subset=['승차_역', '하차_역'])
df_node_score = pd.read_excel(file_path_node_score)

od_weight_dict = df_od.set_index(['승차_역', '하차_역'])['쌍_가중치_정규화'].to_dict()
node_score_dict = df_node_score.set_index('역명')['총점수_정규화'].to_dict()

G_base = nx.Graph()
for _, row in df_edges.iterrows():
    start, end = row['from'], row['to']
    start_pos = (row['from_lon'], row['from_lat'])
    end_pos = (row['to_lon'], row['to_lat'])
    dist = row['distance']
    G_base.add_node(start, pos=start_pos)
    G_base.add_node(end, pos=end_pos)
    G_base.add_edge(start, end, weight=dist)

def heuristic(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

def random_a_star(graph, start, goal, used_edges):
    open_set = []
    heappush(open_set, (0, start))
    came_from = {}
    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start] = 0
    f_score = {node: float('inf') for node in graph.nodes}
    f_score[start] = heuristic(graph.nodes[start]['pos'], graph.nodes[goal]['pos'])

    while open_set:
        _, current = heappop(open_set)
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]

        neighbors = list(graph.neighbors(current))
        random.shuffle(neighbors)
        for neighbor in neighbors:
            edge = tuple(sorted((current, neighbor)))
            if edge in used_edges:
                continue
            tentative_g = g_score[current] + graph[current][neighbor]['weight']
            if tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f_score[neighbor] = tentative_g + heuristic(graph.nodes[neighbor]['pos'], graph.nodes[goal]['pos'])
                heappush(open_set, (f_score[neighbor], neighbor))
    return None

# 2호선 고정 노선
line2_station_names = ['시청', '을지로입구', '을지로3가', '을지로4가', '동대문역사문화공원',
                       '신당', '상왕십리', '왕십리', '한양대', '뚝섬', '성수', '건대입구',
                       '구의', '강변', '잠실나루', '잠실', '잠실새내', '종합운동장', '삼성',
                       '선릉', '역삼', '강남', '교대', '서초', '방배', '사당', '낙성대',
                       '서울대입구', '봉천', '신림', '신대방', '구로디지털단지', '대림',
                       '신도림', '문래', '영등포구청', '당산', '합정', '홍대입구', '신촌',
                       '이대', '아현', '충정로', '시청']
line2_nodes = [name for name in line2_station_names if name in G_base.nodes]
line2_full_path, line2_used_edges = [], set()
for u, v in zip(line2_nodes[:-1], line2_nodes[1:]):
    if not G_base.has_edge(u, v):
        dist = np.linalg.norm(np.array(G_base.nodes[u]['pos']) - np.array(G_base.nodes[v]['pos']))
        G_base.add_edge(u, v, weight=dist)
    line2_used_edges.add(tuple(sorted((u, v))))
    if not line2_full_path or line2_full_path[-1] != u:
        line2_full_path.extend([u, v])
    else:
        line2_full_path.append(v)

custom_terminal_pairs = [
    ('천왕', '지축'),
    ('역곡', '회룡'),
    ('하남검단산', '방화'),
    ('독바위', '신내'),
    ('불암산', '남태령'),
    ('개화', '중앙보훈병원'),
    ('수락산', '마천'),
    ('암사', '모란'),
]

# 적합도 + 길이 패널티 함수
def fitness_path_length_range(individual, min_length=20, max_length=50):
    penalty = 0
    for line_num, path in individual:
        if line_num == 100:
            continue
        length = len(path)
        if length < min_length:
            penalty += (min_length - length) ** 2
        elif length > max_length:
            penalty += (length - max_length) ** 2
    return -penalty

def calculate_combined_fitness(line_paths, od_dict, node_dict):
    od_score = 0
    for _, path in line_paths:
        for i in range(len(path)):
            for j in range(i + 1, len(path)):
                pair = (path[i], path[j])
                rev_pair = (path[j], path[i])
                if pair in od_dict:
                    od_score += od_dict[pair]
                elif rev_pair in od_dict:
                    od_score += od_dict[rev_pair]
    node_score = sum(node_dict[n] for _, path in line_paths for n in path if n in node_dict)
    return od_score + node_score + fitness_path_length_range(line_paths)

# 개체군 생성 & 검사
population, fitness_scores = [], []
for _ in range(1000):
    G = G_base.copy()
    for u, v in G.edges():
        dist = np.linalg.norm(np.array(G.nodes[u]['pos']) - np.array(G.nodes[v]['pos']))
        G[u][v]['weight'] = dist * random.uniform(0.1, 1.9)

    line_paths = [(100, line2_full_path)]
    used_edges_copy = set(line2_used_edges)
    valid_paths = True

    for i, (start, goal) in enumerate(custom_terminal_pairs):
        path = random_a_star(G, start, goal, used_edges_copy)
        if path is None:
            valid_paths = False
            break
        for u, v in zip(path[:-1], path[1:]):
            used_edges_copy.add(tuple(sorted((u, v))))
        line_paths.append((i, path))

    if not valid_paths or len(line_paths) != 9:
        continue

    # 로직
    all_nodes = set(G_base.nodes)
    covered_nodes = set(n for _, p in line_paths for n in p)
    missing_nodes = list(all_nodes - covered_nodes)

    for missing_node in missing_nodes:
        best_increase, best_line_idx, best_insert_pos = float('inf'), None, None
        for i, (line_num, path) in enumerate(line_paths):
            if line_num == 100:
                continue
            for j in range(len(path) - 1):
                u, v = path[j], path[j + 1]
                cur_cost = np.linalg.norm(np.array(G_base.nodes[u]['pos']) - np.array(G_base.nodes[v]['pos']))
                new_cost = (
                    np.linalg.norm(np.array(G_base.nodes[u]['pos']) - np.array(G_base.nodes[missing_node]['pos'])) +
                    np.linalg.norm(np.array(G_base.nodes[missing_node]['pos']) - np.array(G_base.nodes[v]['pos']))
                )
                if new_cost - cur_cost < best_increase:
                    best_increase = new_cost - cur_cost
                    best_line_idx = i
                    best_insert_pos = j + 1
        if best_line_idx is not None:
            line_num, path = line_paths[best_line_idx]
            line_paths[best_line_idx] = (line_num, path[:best_insert_pos] + [missing_node] + path[best_insert_pos:])

    population.append(line_paths)
    fitness_scores.append(calculate_combined_fitness(line_paths, od_weight_dict, node_score_dict))

# 상위 10개 검사
top_10_indices = np.argsort(fitness_scores)[-10:][::-1]

line_colors = {
    1: '#0052A4', 2: '#00A84D', 3: '#EF7C1C', 4: '#00A0DE', 5: '#996CAC',
    6: '#CD7C2F', 7: '#747F00', 8: '#E6186C', 9: '#B7C452'
}

for rank, idx in enumerate(top_10_indices, 1):
    solution = population[idx]
    score = fitness_scores[idx]  # 적합도 점수 가져오기

    fig, ax = plt.subplots(figsize=(6, 6))
    for line_num, path in solution:
        if line_num == 100:
            color = line_colors[2]
            label = 'Line 2'
        else:
            corrected_line = line_num + 1 if line_num < 1 else line_num + 2
            color = line_colors.get(corrected_line, 'gray')
            label = f'Line {corrected_line}'

        for u, v in zip(path[:-1], path[1:]):
            x1, y1 = G_base.nodes[u]['pos']
            x2, y2 = G_base.nodes[v]['pos']
            if u == path[0]:
                ax.plot([x1, x2], [y1, y2], color=color, linewidth=2.5, label=label)
            else:
                ax.plot([x1, x2], [y1, y2], color=color, linewidth=2.5)

    for _, data in G_base.nodes(data=True):
        ax.plot(data['pos'][0], data['pos'][1], 'k.', markersize=3)

    handles, labels = ax.get_legend_handles_labels()
    sorted_labels_handles = sorted(zip(labels, handles), key=lambda x: int(x[0].split()[-1]))
    labels, handles = zip(*sorted_labels_handles)
    ax.legend(handles, labels, loc='best')

    # 점수를 포함한 제목 설정
    plt.title(f"Rank {rank} | Score: {score:.2f}", fontsize=10)
    plt.axis('off')
    plt.show()

def generate_neighbor_soft(individual, graph, num_swaps=2):
    neighbor = copy.deepcopy(individual)
    candidates = [i for i, (line_num, path) in enumerate(neighbor) if line_num != 100 and len(path) > 3]
    if not candidates:
        return neighbor
    idx = random.choice(candidates)
    line_num, path = neighbor[idx]
    for _ in range(num_swaps):
        i = random.randint(1, len(path) - 3)
        if graph.has_edge(path[i-1], path[i+1]) and graph.has_edge(path[i], path[i+2]):
            path[i], path[i+1] = path[i+1], path[i]
    neighbor[idx] = (line_num, path)
    return neighbor

def simulated_annealing(individual, score_func, od_dict, node_dict, graph, T=1.0, alpha=0.995, iterations=500):
    current = copy.deepcopy(individual)
    best = copy.deepcopy(individual)
    best_score = score_func(best, od_dict, node_dict)
    current_score = best_score
    for _ in range(iterations):
        neighbor = generate_neighbor_soft(current, graph)
        neighbor_score = score_func(neighbor, od_dict, node_dict)
        if neighbor_score > current_score:
            current = neighbor
            current_score = neighbor_score
            if neighbor_score > best_score:
                best = neighbor
                best_score = neighbor_score
        else:
            prob = np.exp((neighbor_score - current_score) / T)
            if random.random() < prob:
                current = neighbor
                current_score = neighbor_score
        T *= alpha
    return best, best_score

# 반복 실행 횟수 지정
num_trials = 10
best_overall = None
best_score_overall = -np.inf

for trial in range(num_trials):
    best_result, best_score = simulated_annealing(
        population[top_10_indices[0]],
        calculate_combined_fitness,
        od_weight_dict,
        node_score_dict,
        G_base,
        T=1.0, alpha=0.995, iterations=1000
    )
    print(f"[Trial {trial+1}] Original: {fitness_scores[top_10_indices[0]]:.2f} | Improved: {best_score:.2f}")
    if best_score > best_score_overall:
        best_overall = best_result
        best_score_overall = best_score

# 시각화
fig, ax = plt.subplots(figsize=(6, 6))
for line_num, path in best_overall:
    if line_num == 100:
        color = line_colors[2]
        label = 'Line 2'
    else:
        corrected_line = line_num + 1 if line_num < 1 else line_num + 2
        color = line_colors.get(corrected_line, 'gray')
        label = f'Line {corrected_line}'
    for u, v in zip(path[:-1], path[1:]):
        x1, y1 = G_base.nodes[u]['pos']
        x2, y2 = G_base.nodes[v]['pos']
        ax.plot([x1, x2], [y1, y2], color=color, linewidth=2.5, label=label if u == path[0] else "")
for _, data in G_base.nodes(data=True):
    ax.plot(data['pos'][0], data['pos'][1], 'k.', markersize=3)
ax.set_title(f"Best of {num_trials} Trials | Score: {best_score_overall:.2f}")
ax.legend(loc='best')
plt.axis('off')
plt.show()

개체 개선 알고리즘

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import random
from heapq import heappush, heappop

!pip install openpyxl

file_path_edge = "/content/엣지데이터 신규.xlsx"
file_path_node = "/content/역 경도 위도.xlsx"
file_path_od_weight = "/content/쌍가중치_정규화_OD.xlsx"
file_path_node_score = "/content/총점수_정규화_역주변시설.xlsx"

df_edges = pd.read_excel(file_path_edge)
df_nodes = pd.read_excel(file_path_node, sheet_name="Sheet1")
df_od = pd.read_excel(file_path_od_weight).dropna(subset=['승차_역', '하차_역'])
df_node_score = pd.read_excel(file_path_node_score)

od_weight_dict = df_od.set_index(['승차_역', '하차_역'])['쌍_가중치_정규화'].to_dict()
node_score_dict = df_node_score.set_index('역명')['총점수_정규화'].to_dict()

G_base = nx.Graph()
for _, row in df_edges.iterrows():
    u, v = row['from'], row['to']
    pos_u = (row['from_lon'], row['from_lat'])
    pos_v = (row['to_lon'], row['to_lat'])
    dist = row['distance']
    G_base.add_node(u, pos=pos_u)
    G_base.add_node(v, pos=pos_v)
    G_base.add_edge(u, v, weight=dist)

def heuristic(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

def random_a_star(graph, start, goal, used_edges):
    open_set = []
    heappush(open_set, (0, start))
    came_from = {}
    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start] = 0
    f_score = {node: float('inf') for node in graph.nodes}
    f_score[start] = heuristic(graph.nodes[start]['pos'], graph.nodes[goal]['pos'])

    while open_set:
        _, current = heappop(open_set)
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]

        neighbors = list(graph.neighbors(current))
        random.shuffle(neighbors)
        for neighbor in neighbors:
            edge = tuple(sorted((current, neighbor)))
            if edge in used_edges:
                continue
            tentative_g = g_score[current] + graph[current][neighbor]['weight']
            if tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f_score[neighbor] = tentative_g + heuristic(graph.nodes[neighbor]['pos'], graph.nodes[goal]['pos'])
                heappush(open_set, (f_score[neighbor], neighbor))
    return None

line2_station_names = ['시청', '을지로입구', '을지로3가', '을지로4가', '동대문역사문화공원',
    '신당', '상왕십리', '왕십리', '한양대', '뚝섬', '성수', '건대입구', '구의', '강변', '잠실나루',
    '잠실', '잠실새내', '종합운동장', '삼성', '선릉', '역삼', '강남', '교대', '서초', '방배',
    '사당', '낙성대', '서울대입구', '봉천', '신림', '신대방', '구로디지털단지', '대림',
    '신도림', '문래', '영등포구청', '당산', '합정', '홍대입구', '신촌', '이대', '아현', '충정로', '시청']
line2_nodes = [s for s in line2_station_names if s in G_base.nodes]
line2_path = []
line2_edges = set()
for u, v in zip(line2_nodes[:-1], line2_nodes[1:]):
    if not G_base.has_edge(u, v):
        dist = np.linalg.norm(np.array(G_base.nodes[u]['pos']) - np.array(G_base.nodes[v]['pos']))
        G_base.add_edge(u, v, weight=dist)
    line2_edges.add(tuple(sorted((u, v))))
    if not line2_path or line2_path[-1] != u:
        line2_path.extend([u, v])
    else:
        line2_path.append(v)

def fitness_path_length_range(individual, min_length=20, max_length=50):
    penalty = 0
    for line_num, path in individual:
        if line_num == 100: continue
        if len(path) < min_length:
            penalty += (min_length - len(path))**2
        elif len(path) > max_length:
            penalty += (len(path) - max_length)**2
    return -penalty

# 각도 변화량 패널티
def angle_between_nodes(pos_u, pos_v, pos_w):
    vec1 = np.array(pos_u) - np.array(pos_v)
    vec2 = np.array(pos_w) - np.array(pos_v)
    cosine = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    cosine = np.clip(cosine, -1.0, 1.0)
    return np.degrees(np.arccos(cosine))

def smooth_angle_penalty(line_paths, graph):
    penalty = 0
    for _, path in line_paths:
        if len(path) < 4:
            continue

        # 단일 각도 기준 (급격한 꺾임 탐지)
        for i in range(1, len(path) - 1):
            u, v, w = path[i - 1], path[i], path[i + 1]
            angle = angle_between_nodes(graph.nodes[u]['pos'], graph.nodes[v]['pos'], graph.nodes[w]['pos'])
            if angle < 90:
                penalty += (90 - angle)

        # 연속 각도의 변화량 기준
        for i in range(1, len(path) - 2):
            u1, v1, w1 = path[i - 1], path[i], path[i + 1]
            u2, v2, w2 = path[i], path[i + 1], path[i + 2]

            angle1 = angle_between_nodes(graph.nodes[u1]['pos'], graph.nodes[v1]['pos'], graph.nodes[w1]['pos'])
            angle2 = angle_between_nodes(graph.nodes[u2]['pos'], graph.nodes[v2]['pos'], graph.nodes[w2]['pos'])

            penalty += (abs(angle1 - angle2))

    return -penalty

# 35도 이하는 생성 금지
def is_individual_angle_valid(individual, graph, min_angle=35):
    for _, path in individual:
        if len(path) < 3:
            continue
        for i in range(1, len(path) - 1):
            u, v, w = path[i - 1], path[i], path[i + 1]
            angle = angle_between_nodes(graph.nodes[u]['pos'], graph.nodes[v]['pos'], graph.nodes[w]['pos'])
            if angle < min_angle:
                return False
    return True


# 종합 적합도 함수
def calculate_combined_fitness(line_paths, od_dict, node_dict,
                                alpha=1.0, beta=2.5, gamma=0.1, delta=0.013):
    od_score = 0
    for _, path in line_paths:
        for i in range(len(path)):
            for j in range(i + 1, len(path)):
                pair = (path[i], path[j])
                rev_pair = (path[j], path[i])
                if pair in od_dict:
                    od_score += od_dict[pair]
                elif rev_pair in od_dict:
                    od_score += od_dict[rev_pair]

    node_score = sum(node_dict.get(n, 0) for _, path in line_paths for n in path)
    length_penalty = fitness_path_length_range(line_paths)
    angle_penalty_val = smooth_angle_penalty(line_paths, G_base)

    return alpha * od_score + beta * node_score + gamma * length_penalty + delta * angle_penalty_val

### 개선 할때 60미만 각도 우선 추출
def improve_individual_by_angle_with_batch_insertion(individual, graph, min_angle=60, batch_size=4):
    improved = [ (lnum, path[:]) for lnum, path in individual ]
    candidates = []

    # 각도 min_angle 미만인 지점 수집
    for line_idx, (lnum, path) in enumerate(improved):
        if lnum == 100 or len(path) < 3:
            continue
        for i in range(1, len(path) - 1):
            u, v, w = path[i - 1], path[i], path[i + 1]
            angle = angle_between_nodes(graph.nodes[u]['pos'], graph.nodes[v]['pos'], graph.nodes[w]['pos'])
            if angle < min_angle:
                candidates.append((line_idx, i, v))

    if not candidates:
        return improved

    selected = random.sample(candidates, min(batch_size, len(candidates)))
    removed_nodes = []

    for line_idx, del_idx, bad_node in selected:
        lnum, path = improved[line_idx]
        if del_idx < len(path):
            new_path = path[:del_idx] + path[del_idx+1:]
            improved[line_idx] = (lnum, new_path)
            removed_nodes.append(bad_node)

    for bad_node in removed_nodes:
        best_dist = float('inf')
        best_insert = None

        for i, (lnum2, path2) in enumerate(improved):
            if len(path2) < 2:
                continue
            for j in range(len(path2) - 1):
                a, b = path2[j], path2[j + 1]
                pos_a = graph.nodes[a]['pos']
                pos_b = graph.nodes[b]['pos']
                pos_v = graph.nodes[bad_node]['pos']
                dist = np.linalg.norm(np.array(pos_v) - (np.array(pos_a) + np.array(pos_b)) / 2)
                if dist < best_dist:
                    best_dist = dist
                    best_insert = (i, j + 1)

        if best_insert:
            i, insert_pos = best_insert
            lnum2, path2 = improved[i]
            if bad_node not in path2:
                a = path2[insert_pos - 1]
                b = path2[insert_pos]
                if not graph.has_edge(a, bad_node):
                    dist1 = np.linalg.norm(np.array(graph.nodes[a]['pos']) - np.array(graph.nodes[bad_node]['pos']))
                    graph.add_edge(a, bad_node, weight=dist1)
                if not graph.has_edge(bad_node, b):
                    dist2 = np.linalg.norm(np.array(graph.nodes[bad_node]['pos']) - np.array(graph.nodes[b]['pos']))
                    graph.add_edge(bad_node, b, weight=dist2)

                new_path2 = path2[:insert_pos] + [bad_node] + path2[insert_pos:]
                improved[i] = (lnum2, new_path2)

    return improved


def improve_by_batch_safe_granular(individual, graph, od_dict, node_dict, iterations=10, batch_size=4, min_angle=60):
    current = [ (lnum, path[:]) for lnum, path in individual ]
    current_score = calculate_combined_fitness(current, od_dict, node_dict)

    for _ in range(iterations):
        # 꺾인 점 후보 수집
        candidates = []
        for line_idx, (lnum, path) in enumerate(current):
            if lnum == 100 or len(path) < 3:
                continue
            for i in range(1, len(path) - 1):
                u, v, w = path[i - 1], path[i], path[i + 1]
                angle = angle_between_nodes(graph.nodes[u]['pos'], graph.nodes[v]['pos'], graph.nodes[w]['pos'])
                if angle < min_angle:
                    candidates.append((line_idx, i, v))

        if not candidates:
            break  # 개선할 점이 없음

        selected = random.sample(candidates, min(batch_size, len(candidates)))

        for line_idx, del_idx, bad_node in selected:
            # 현재 상태를 복사
            trial = [ (lnum, path[:]) for lnum, path in current ]
            lnum, path = trial[line_idx]

            # 노드 제거
            if del_idx >= len(path): continue
            path = path[:del_idx] + path[del_idx+1:]
            trial[line_idx] = (lnum, path)

            # 삽입 위치 탐색
            best_dist = float('inf')
            best_insert = None
            for i, (lnum2, path2) in enumerate(trial):
                if len(path2) < 2 or bad_node in path2: continue
                for j in range(len(path2) - 1):
                    a, b = path2[j], path2[j + 1]
                    pos_a = graph.nodes[a]['pos']
                    pos_b = graph.nodes[b]['pos']
                    pos_v = graph.nodes[bad_node]['pos']
                    dist = np.linalg.norm(np.array(pos_v) - (np.array(pos_a) + np.array(pos_b)) / 2)
                    if dist < best_dist:
                        best_dist = dist
                        best_insert = (i, j + 1)

            # 삽입 시도
            if best_insert:
                i, insert_pos = best_insert
                lnum2, path2 = trial[i]
                a = path2[insert_pos - 1]
                b = path2[insert_pos]
                if not graph.has_edge(a, bad_node):
                    dist1 = np.linalg.norm(np.array(graph.nodes[a]['pos']) - np.array(graph.nodes[bad_node]['pos']))
                    graph.add_edge(a, bad_node, weight=dist1)
                if not graph.has_edge(bad_node, b):
                    dist2 = np.linalg.norm(np.array(graph.nodes[bad_node]['pos']) - np.array(graph.nodes[b]['pos']))
                    graph.add_edge(bad_node, b, weight=dist2)
                new_path2 = path2[:insert_pos] + [bad_node] + path2[insert_pos:]
                trial[i] = (lnum2, new_path2)

            # 개선 여부 판단
            trial_score = calculate_combined_fitness(trial, od_dict, node_dict)
            if trial_score >= current_score:
                current = trial
                current_score = trial_score


    return current



custom_terminal_pairs = [
    ('천왕', '지축'), ('역곡', '회룡'), ('하남검단산', '방화'), ('독바위', '신내'),
    ('불암산', '남태령'), ('개화', '중앙보훈병원'), ('수락산', '마천'), ('암사', '모란')
]

population = []
fitness_scores = []
for _ in range(100000):
    G = G_base.copy()
    for u, v in G.edges():
        dist = np.linalg.norm(np.array(G.nodes[u]['pos']) - np.array(G.nodes[v]['pos']))
        G[u][v]['weight'] = dist * random.uniform(0.1, 1.9)

    used_edges = set(line2_edges)
    line_paths = [(100, line2_path)]
    valid = True

    for i, (start, goal) in enumerate(custom_terminal_pairs):
        path = random_a_star(G, start, goal, used_edges)
        if path is None:
            valid = False
            break
        for u, v in zip(path[:-1], path[1:]):
            used_edges.add(tuple(sorted((u, v))))
        line_paths.append((i, path))

    if not valid or len(line_paths) != 9:
        continue


    all_nodes = set(G_base.nodes)
    covered = set(n for _, p in line_paths for n in p)
    missing = list(all_nodes - covered)

    # 미포함 노드 포함시키는 로직
    for mn in missing:
        best_increase = float('inf')
        for i, (lnum, path) in enumerate(line_paths):
            if lnum == 100: continue
            for j in range(len(path)-1):
                u, v = path[j], path[j+1]
                cost_now = np.linalg.norm(np.array(G_base.nodes[u]['pos']) - np.array(G_base.nodes[v]['pos']))
                cost_new = np.linalg.norm(np.array(G_base.nodes[u]['pos']) - np.array(G_base.nodes[mn]['pos'])) + \
                           np.linalg.norm(np.array(G_base.nodes[mn]['pos']) - np.array(G_base.nodes[v]['pos']))
                if cost_new - cost_now < best_increase:
                    best = (i, j+1)
                    best_increase = cost_new - cost_now
        if best_increase < float('inf'):
            i, pos = best
            lnum, path = line_paths[i]
            line_paths[i] = (lnum, path[:pos] + [mn] + path[pos:])

    # 추가: 각도 조건 검사 (60도 미만 각도 존재 시 개체 폐기)
    if not is_individual_angle_valid(line_paths, G_base):
        continue

    population.append(line_paths)
    fitness_scores.append(calculate_combined_fitness(line_paths, od_weight_dict, node_score_dict))


top_10_indices = np.argsort(fitness_scores)[-10:][::-1]

line_colors = {
    1: '#0052A4', 2: '#00A84D', 3: '#EF7C1C', 4: '#00A0DE', 5: '#996CAC',
    6: '#CD7C2F', 7: '#747F00', 8: '#E6186C', 9: '#B7C452'
}


for rank, idx in enumerate(top_10_indices, 1):
    solution = population[idx]
    score = fitness_scores[idx]

    fig, ax = plt.subplots(figsize=(6, 6))
    for line_num, path in solution:
        if line_num == 100:
            color = line_colors[2]
            label = 'Line 2'
        else:
            corrected_line = line_num + 1 if line_num < 1 else line_num + 2
            color = line_colors.get(corrected_line, 'gray')
            label = f'Line {corrected_line}'

        for u, v in zip(path[:-1], path[1:]):
            x1, y1 = G_base.nodes[u]['pos']
            x2, y2 = G_base.nodes[v]['pos']
            if u == path[0]:
                ax.plot([x1, x2], [y1, y2], color=color, linewidth=2.5, label=label)
            else:
                ax.plot([x1, x2], [y1, y2], color=color, linewidth=2.5)

    for _, data in G_base.nodes(data=True):
        ax.plot(data['pos'][0], data['pos'][1], 'k.', markersize=3)

    handles, labels = ax.get_legend_handles_labels()
    sorted_labels_handles = sorted(zip(labels, handles), key=lambda x: int(x[0].split()[-1]))
    labels, handles = zip(*sorted_labels_handles)
    ax.legend(handles, labels, loc='best')

    plt.title(f"Rank {rank} | Score: {score:.2f}", fontsize=10)
    plt.axis('off')
    plt.show()


# top 1 개체 선택
original = population[top_10_indices[0]]

# 개선: 점수 기반 유지
improved = improve_by_batch_safe_granular(
    original, G_base, od_weight_dict, node_score_dict,
    iterations=10000,
    batch_size=4,
    min_angle=60
)


# 점수 비교
before_score = calculate_combined_fitness(original, od_weight_dict, node_score_dict)
after_score = calculate_combined_fitness(improved, od_weight_dict, node_score_dict)

print(f"before score: {before_score:.2f} → after score: {after_score:.2f}")

# 개선된 개체 시각화
fig, ax = plt.subplots(figsize=(6, 6))
for line_num, path in improved:
    if line_num == 100:
        color = line_colors[2]
        label = 'Line 2'
    else:
        corrected_line = line_num + 1 if line_num < 1 else line_num + 2
        color = line_colors.get(corrected_line, 'gray')
        label = f'Line {corrected_line}'

    for u, v in zip(path[:-1], path[1:]):
        x1, y1 = G_base.nodes[u]['pos']
        x2, y2 = G_base.nodes[v]['pos']
        if u == path[0]:
            ax.plot([x1, x2], [y1, y2], color=color, linewidth=2.5, label=label)
        else:
            ax.plot([x1, x2], [y1, y2], color=color, linewidth=2.5)

for _, data in G_base.nodes(data=True):
    ax.plot(data['pos'][0], data['pos'][1], 'k.', markersize=3)

handles, labels = ax.get_legend_handles_labels()
sorted_labels_handles = sorted(zip(labels, handles), key=lambda x: int(x[0].split()[-1]))
labels, handles = zip(*sorted_labels_handles)
ax.legend(handles, labels, loc='best')

plt.title(f"After Score: {after_score:.2f}", fontsize=10)
plt.axis('off')
plt.show()